|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 3:</h2>|<h1>PagedAttention<h1>|
|<h2>Section:</h2>|<h1>Sharing blocks<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: build automatic prefix caching<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import hashlib
from collections import OrderedDict

import numpy as np

rng = np.random.default_rng(0)

Build automatic prefix caching.

No tensors: the interesting part is entirely in the bookkeeping, and the
bookkeeping has one trap in it that produces fluent, confident, wrong output
if you get it wrong.

In [ ]:
### run this cell

BLOCK = 16

SYSTEM_A = list(rng.integers(0, 50000, size=96))   # 6 blocks
SYSTEM_B = list(rng.integers(0, 50000, size=96))   # a different one

def request(system, n_question=40):
  return system + list(rng.integers(0, 50000, size=n_question))

print(f'system prompts are {len(SYSTEM_A)//BLOCK} blocks each')

# Exercise 1: hash the blocks

One hash per full block, so two requests that begin the same way produce the
same hashes for as far as they agree.

In [ ]:
def block_hashes(tokens, block_size=BLOCK):
  """One hash per FULL block. Each hash covers every token up to and
  including that block, because block k's K and V depend on blocks 0..k."""
  out, h = [], hashlib.sha256()
  for i in range(0, len(tokens) - block_size + 1, block_size):
    h = h.copy()
    h.update(np.array(tokens[i:i+block_size], dtype=np.int64).tobytes())
    out.append(h.hexdigest()[:16])
  return out

a = block_hashes(request(SYSTEM_A))
b = block_hashes(request(SYSTEM_A))
c = block_hashes(request(SYSTEM_B))

print('two requests with system A share', sum(x==y for x,y in zip(a,b)), 'block hashes')
print('A against B share              ', sum(x==y for x,y in zip(a,c)))

# Exercise 2: the trap

Take the same sixteen tokens and put them after two different system prompts.
Do they belong in the same cache entry?

Think about what a KV block physically contains before you answer.

In [ ]:
shared_tail = list(rng.integers(0, 50000, size=BLOCK))

one = SYSTEM_A + shared_tail        # tail follows system A
two = SYSTEM_B + shared_tail        # the SAME tail, following system B

h1, h2 = block_hashes(one), block_hashes(two)

print('the two requests end with identical tokens:', one[-BLOCK:] == two[-BLOCK:])
print('and the hash of that last block matches:   ', h1[-1] == h2[-1])
print('\nIt must NOT match. The K and V of those tokens depend on everything')
print('before them, and the two prefixes are different.')

# Exercise 3: the cache

A hash to block-id map, an LRU eviction order, and a lookup that returns the
longest cached **prefix**.

In [ ]:
class PrefixCache:
  def __init__(self, capacity_blocks):
    self.cap   = capacity_blocks
    self.cache = OrderedDict()          # hash -> physical block id
    self.hits = self.misses = 0
    self.next_block = 0

  def lookup(self, hashes):
    """Return the block ids for the longest cached PREFIX of `hashes`."""
    out = []
    for h in hashes:
      if h not in self.cache:
        break                            # a gap ends the prefix
      self.cache.move_to_end(h)
      out.append(self.cache[h])
    self.hits   += len(out)
    self.misses += len(hashes) - len(out)
    return out

  def insert(self, hashes):
    for h in hashes:
      if h in self.cache:
        self.cache.move_to_end(h); continue
      self.cache[h] = self.next_block; self.next_block += 1
      if len(self.cache) > self.cap:
        self.cache.popitem(last=False)   # least recently used

pc = PrefixCache(64)
r1 = block_hashes(request(SYSTEM_A)); pc.insert(r1)
r2 = block_hashes(request(SYSTEM_A))
print(f'second request hit {len(pc.lookup(r2))} of {len(r2)} blocks')

# Exercise 4: run a workload through it

Five popular system prompts and a long tail of unique ones, which is roughly
what a real deployment looks like. Sweep the capacity.

In [ ]:
systems = [list(rng.integers(0, 50000, size=96)) for _ in range(5)]
weights = np.array([.4,.2,.15,.15,.1])

def trace(n=3000, popular_frac=0.8):
  out = []
  for _ in range(n):
    if rng.random() < popular_frac:
      sysp = systems[rng.choice(len(systems), p=weights)]
    else:
      sysp = list(rng.integers(0, 50000, size=96))    # a unique prompt
    out.append(block_hashes(request(sysp)))
  return out

stream = trace()
print(f"{'blocks of cache':>16} {'hit rate':>9} {'system prompts held':>21}")
for cap in (6, 12, 30, 60, 120, 600):
  pc = PrefixCache(cap)
  for hs in stream:
    pc.lookup(hs); pc.insert(hs)
  rate = pc.hits/(pc.hits+pc.misses)
  print(f'{cap:>16} {100*rate:>8.1f}% {cap/6:>20.0f}')

### The three things that make this correct

**The hash chains.** Block `k`'s hash covers tokens 0 through the end of
block `k`, not just its own sixteen. Hash the block alone and Exercise 2
hands two different requests the same cache entry, which returns K and V
computed against the wrong prefix. The output stays fluent and becomes
wrong, which is the worst failure mode available.

**The lookup stops at the first gap.** You cannot use block 4 without
block 3, because there is nothing to attend it against. A cache that
returned a set instead of a prefix would report a lovely hit rate and
produce nonsense.

**Only full blocks are hashed.** A partial block is still being written,
so its contents are not final and hashing it would cache a value about
to change.

### And the number

The hit rate saturates almost immediately. Five system prompts account for
most of the traffic, so a cache with room for a few of them captures
nearly everything a cache a hundred times bigger would.

Which is the useful engineering conclusion: prefix caching is not a
memory-hungry feature. It is cheap, and it is why teams find they can
afford a far longer system prompt than they budgeted for.

    ./vc guide 9